In [20]:
%pip install -q pandas numpy ollama


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


# Imports

In [21]:
import time

import pandas as pd
from ollama import chat
from pydantic import BaseModel, Field

from src.constants import Column, INPUT_DIR, OUTPUT_DIR

# Config

In [22]:
MODEL = "qwen3:14b"
MAX_REVIEWS = 10  # Für den vollständigen Lauf auf None setzen
NUM_CTX = 2048
NUM_PREDICT = 160

## Load dataset

In [23]:
electronics_df = pd.read_csv(INPUT_DIR / "electronics_raw.csv")

if MAX_REVIEWS is not None:
    electronics_df = electronics_df.head(MAX_REVIEWS).copy()
electronics_df = electronics_df.reset_index(drop=True)

result_df = pd.DataFrame()

# Enable LLM

## Setup output schema

In [24]:
class Output(BaseModel):
    readability: float = Field(ge=0, le=1)
    subjectivity: float = Field(ge=0, le=1)
    internal_consistency: float = Field(ge=0, le=1)

    adjective_count: int
    verb_count: int
    superlative_count: int

In [25]:
MASTER_PROMPT = """
Code an online product review for an academic study. Never judge whether it
is fake, genuine, human-written, or generated.

Return decimal scores from 0.0 to 1.0 for:

subjectivity: How strongly the text depends on personal judgment, preference,
feeling, or unverifiable evaluation. Product specifications and externally
verifiable claims are objective even when positive or negative. First-person
pronouns alone do not make a statement subjective. Ignore the star rating.
Anchors: 0.0 only factual; 0.25 mostly factual with minor evaluation; 0.5 an
even mix; 0.75 mostly opinions or feelings; 1.0 entirely personal/evaluative.
Examples: 'The battery has 5,000 mAh and a two-year warranty.' -> 0.0;
'I absolutely love this phone; it feels perfect to me.' -> 1.0.

readability: Ease of understanding based on grammar, clarity, wording, and
structure. 0.0 severely broken/incoherent; 0.5 understandable with noticeable
problems; 1.0 effortless and coherent. Ignore sentiment, rating, and truth.
Examples: 'Product because working no, yesterday maybe.' -> 0.0; 'The product
works good, but instructions is difficult.' -> 0.5; 'The product works well,
but the instructions are difficult to follow.' -> 1.0.

internal_consistency: Agreement between the review's overall opinion and the
star rating, where 1 star is the worst and 5 stars is the best rating.
0.0 contradicts it; 0.5 is mixed/ambiguous; 1.0 clearly matches it.
Examples: 'Excellent; it works perfectly.' + 5 stars -> 1.0; the same text
+ 1 star -> 0.0; 'Some strengths but several problems.' + 3 stars -> 1.0.

Also count tokens using ordinary English tokenization; count each token at
most once per category:
- adjective_count: adjectives, including comparative and superlative forms
- verb_count: lexical and auxiliary verbs
- superlative_count: superlative adjectives and adverbs

Return only valid JSON matching the schema; no explanation or Markdown.
"""

In [26]:
def analyze_review(review_text, rating):
    response = chat(
        model=MODEL,
        messages=[
            {
                "role": "system",
                "content": MASTER_PROMPT,
            },
            {
                "role": "user",
                "content": f"Review: {review_text}\nStar rating: {rating}",
            },
        ],
        keep_alive=-1,
        think=False,
        format=Output.model_json_schema(),
        options={
            "temperature": 0,
            "num_ctx": NUM_CTX,
            "num_predict": NUM_PREDICT,
        },
    )

    return Output.model_validate_json(
        response.message.content
    )

In [27]:
## Test with one single dataset
print(analyze_review("This is the biggest TV I saw in my entire life. It's absolutely fantastic.", 2))

readability=0.75 subjectivity=1.0 internal_consistency=0.0 adjective_count=3 verb_count=3 superlative_count=1


## Apply to columns

In [28]:
def analyze_all_reviews():
    total_start = time.perf_counter()
    processing_times = []

    for count, (index, row) in enumerate(electronics_df.iterrows(), start=1):
        review_start = time.perf_counter()

        try:
            result = analyze_review(
                review_text=row[Column.TEXT],
                rating=row[Column.RATING]
            )

            result_df.loc[index, Column.ID] = row[Column.ID]

            result_df.loc[index, Column.SUBJECTIVITY] = result.subjectivity
            result_df.loc[index, Column.READABILITY_LLM] = result.readability
            result_df.loc[index, Column.INTERNAL_CONSISTENCY] = result.internal_consistency

            result_df.loc[index, Column.ADJECTIVE_COUNT] = result.adjective_count
            result_df.loc[index, Column.VERB_COUNT] = result.verb_count
            result_df.loc[index, Column.SUPERLATIVE_COUNT] = result.superlative_count

            review_time = time.perf_counter() - review_start
            processing_times.append(review_time)

            if count % 10 == 0:
                avg_time = sum(processing_times) / len(processing_times)
                estimated_remaining = avg_time * (len(electronics_df) - count)

                print(
                    f"{count}/{len(electronics_df)} verarbeitet | "
                    f"Ø {avg_time:.2f} s/Review | "
                    f"Restzeit ca. {estimated_remaining / 60:.1f} min"
                )

        except Exception as e:
            print(f"Fehler bei Zeile {index}: {e}")

    total_time = time.perf_counter() - total_start

    print(f"\nGesamtzeit: {total_time / 60:.2f} Minuten")

    if processing_times:
        avg_time = sum(processing_times) / len(processing_times)
        print(f"Durchschnittliche Zeit pro erfolgreichem Review: {avg_time:.2f} Sekunden")


analyze_all_reviews()

10/10 verarbeitet | Ø 6.40 s/Review | Restzeit ca. 0.0 min

Gesamtzeit: 1.07 Minuten
Durchschnittliche Zeit pro erfolgreichem Review: 6.40 Sekunden


# Save to CSV

In [29]:
result_df.to_csv(
    OUTPUT_DIR / f"{MODEL}_results.csv",
    index=False, 
    encoding="utf-8"
)